In [1]:
import math
import csv
import numpy as np

class NeuralNetwork:
    def __init__(self, activationFunc, deactivationFunc, hiddenWeights: list, outputWeights: list, requestedEra: int):
        self.requestedEra = requestedEra
        self.hiddenWeights = hiddenWeights
        self.outputWeights = outputWeights
        self.activationFunc = activationFunc
        self.deactivationFunc = deactivationFunc

        self.MseNumerator = 0.0
        self.CurrentMSE = 0.0

        self.E = 0.1     # Скорость обучения
        self.a = 0.1     # Момент

        self.dataset = []
        self.normalized_dataset = []
        self.answersList = []
        
        # ПРАВИЛЬНАЯ инициализация дельт
        self.hiddenWeightDeltas = [[0.0, 0.0, 0.0] for _ in range(3)]
        self.outputWeightDeltas = [0.0, 0.0, 0.0]
        
        # Для нормализации
        self.input_mins = None
        self.input_maxs = None
        self.output_min = None
        self.output_max = None

    def readFromCsv(self, datasetPath: str):
        with open(datasetPath, 'r') as csvDataset:
            csvDatasetReader = csv.reader(csvDataset, delimiter='\t')
            for row in csvDatasetReader:
                self.dataset.append([float(row[0]), float(row[1]), float(row[2]), float(row[3])])
        
        # Нормализация данных
        self._normalizeDataset()

    def _normalizeDataset(self):
        """Нормализация входов и выходов в диапазон [0.1, 0.9]"""
        if not self.dataset:
            return
            
        # Отделяем входы и выходы
        inputs = np.array([row[:3] for row in self.dataset])
        outputs = np.array([row[3] for row in self.dataset])
        
        # Нормализуем входы в [0.1, 0.9]
        self.input_mins = np.min(inputs, axis=0)
        self.input_maxs = np.max(inputs, axis=0)
        
        # Нормализуем выходы в [0.1, 0.9]
        self.output_min = np.min(outputs)
        self.output_max = np.max(outputs)
        
        # Применяем нормализацию
        self.normalized_dataset = []
        for row in self.dataset:
            normalized_inputs = [
                0.1 + 0.8 * (row[i] - self.input_mins[i]) / (self.input_maxs[i] - self.input_mins[i]) 
                for i in range(3)
            ]
            normalized_output = 0.1 + 0.8 * (row[3] - self.output_min) / (self.output_max - self.output_min)
            self.normalized_dataset.append(normalized_inputs + [normalized_output])
        
        print(f"Данные нормализованы. Выходы в диапазоне: {self.output_min:.2f} - {self.output_max:.2f} -> 0.1 - 0.9")

    def denormalize_output(self, normalized_output):
        """Денормализация выхода"""
        if self.output_max == self.output_min:
            return normalized_output
        return self.output_min + (normalized_output - 0.1) * (self.output_max - self.output_min) / 0.8

    def train(self):
        era_mse = []
        for era in range(self.requestedEra):
            self.MseNumerator = 0.0
            self.answersList = []
            
            for fullTrainSet in self.normalized_dataset:
                ideal = fullTrainSet[-1]
                trainSet = fullTrainSet[0:-1]

                # Прямое распространение
                hiddenNeuronInputsValues = self._calculateHiddenInputsValues(trainSet)
                hiddenNeuronOutputsValues = self._calculateHiddenOutputsValues(hiddenNeuronInputsValues)
                output = self._calculateOutput(hiddenNeuronOutputsValues)

                # Денормализованный вывод для MSE
                denormalized_output = self.denormalize_output(output)
                denormalized_ideal = self.denormalize_output(ideal)
                
                self.answersList.append(denormalized_output)
                self.MseNumerator += ((denormalized_ideal - denormalized_output) ** 2)

                # Обратное распространение
                output_error = ideal - output
                dO = output_error * self.deactivationFunc(output)
                # Вычисление dH
                dH = []
                for i in range(len(hiddenNeuronOutputsValues)):
                    dH_i = self.deactivationFunc(hiddenNeuronOutputsValues[i]) * dO * self.outputWeights[i]
                    dH.append(dH_i)

                # Обновление весов
                self.outputWeightDeltas = self._calculateOutputWeightDeltas(dO, hiddenNeuronOutputsValues)
                self.hiddenWeightDeltas = self._calculateHiddenWeightDeltas(trainSet, dH)

                self.hiddenWeights = self._calculateNewHiddenWeigths(self.hiddenWeights, self.hiddenWeightDeltas)
                self.outputWeights = self._calculateNewOutputWeigths(self.outputWeights, self.outputWeightDeltas)

            # Вычисление MSE для эпохи
            epoch_mse = self.MseNumerator / len(self.dataset)
            era_mse.append(epoch_mse)
            
            if era % 100 == 0:
                print(f"Эпоха {era}, MSE: {epoch_mse:.6f}")
                
            # Ранняя остановка
            if epoch_mse < 0.01:
                print(f"Обучение завершено на эпохе {era}")
                break
        
        return era_mse

    def _calculateHiddenInputsValues(self, trainSet: list) -> list:
        result = []
        for weights in self.hiddenWeights:
            halfResult = 0.0
            for i in range(len(weights)):
                halfResult += trainSet[i] * weights[i]
            result.append(halfResult)
        return result

    def _calculateHiddenOutputsValues(self, inputs: list) -> list:
        result = []
        for inputValue in inputs:
            # Ограничиваем вход для избежания переполнения
            if inputValue > 100:
                inputValue = 100
            elif inputValue < -100:
                inputValue = -100
            result.append(self.activationFunc(inputValue))
        return result

    def _calculateOutput(self, hiddenOutputsValues: list) -> float:
        result = 0.0
        for i in range(len(hiddenOutputsValues)):
            result += hiddenOutputsValues[i] * self.outputWeights[i]
        return result

    def _calculateOutputWeightDeltas(self, dO: float, hiddenNeuronOutputsValues: list) -> list:
        outputDeltas = []
        for i in range(len(hiddenNeuronOutputsValues)):
            outputGradient = dO * hiddenNeuronOutputsValues[i]
            previousOutputDelta = self.outputWeightDeltas[i]
            delta = self.E * outputGradient + self.a * previousOutputDelta
            outputDeltas.append(delta)
        return outputDeltas

    def _calculateHiddenWeightDeltas(self, trainSet: list, dH: list) -> list:
        hiddenDeltas = []
        for i in range(len(dH)):
            deltasOfLine = []
            for j in range(len(trainSet)):
                hiddenGradient = trainSet[j] * dH[i]
                previousHiddenDelta = self.hiddenWeightDeltas[i][j]
                delta = self.E * hiddenGradient + self.a * previousHiddenDelta
                deltasOfLine.append(delta)
            hiddenDeltas.append(deltasOfLine)
        return hiddenDeltas

    def _calculateNewHiddenWeigths(self, currentWeights: list, deltas: list):
        newWeights = []
        for i in range(len(currentWeights)):
            lineOfNewWeights = []
            for j in range(len(currentWeights[i])):
                lineOfNewWeights.append(currentWeights[i][j] + deltas[i][j])
            newWeights.append(lineOfNewWeights)
        return newWeights

    def _calculateNewOutputWeigths(self, currentWeights: list, deltas: list):
        newWeights = []
        for i in range(len(currentWeights)):
            newWeights.append(currentWeights[i] + deltas[i])
        return newWeights

In [2]:
# Активационные функции
def f_sig(x):
    # Ограничиваем вход для избежания переполнения
    if x > 100:
        return 1.0
    elif x < -100:
        return 0.0
    return 1 / (1 + math.exp(-x))

def anti_f_sig(x):
    # Производная сигмоиды
    return x * (1 - x)

# # Инициализация СЛУЧАЙНЫМИ МАЛЕНЬКИМИ весами
# np.random.seed(42)
# hiddenWeights = np.random.uniform(-0.5, 0.5, (3, 3)).tolist()
# outputWeights = np.random.uniform(-0.5, 0.5, 3).tolist()

hiddenWeights = [[0.1, 0.9, 0.31], [0.4, 0.7, 0.11], [0.3, 0.2, 0.27]]
# hiddenWeights = [[1, 1, 6], [3, 7, 7], [5, 6, 0]]

outputWeights = [0.47, 0.51, 0.67]
# outputWeights = [1, 4, 6]

print("Начальные веса:")
print("Hidden:", hiddenWeights)
print("Output:", outputWeights)

#RELU
relu = lambda x: max(0, x)
anti_relu = lambda x: x * 1

# myAI = NeuralNetwork(f_sig, anti_f_sig, hiddenWeights, outputWeights, 1000)
myAI = NeuralNetwork(relu, anti_relu, hiddenWeights, outputWeights, 1000)
# myAI.readFromCsv("test_data_sorted_cutted.csv")
myAI.readFromCsv("discriminant_data.csv")
mse_history = myAI.train()

Начальные веса:
Hidden: [[0.1, 0.9, 0.31], [0.4, 0.7, 0.11], [0.3, 0.2, 0.27]]
Output: [0.47, 0.51, 0.67]
Данные нормализованы. Выходы в диапазоне: -91.83 - 23.24 -> 0.1 - 0.9
Эпоха 0, MSE: 1544.754423
Эпоха 100, MSE: 1117.775708
Эпоха 200, MSE: 1227.358738
Эпоха 300, MSE: 1641.863303
Эпоха 400, MSE: 1588.931065
Эпоха 500, MSE: 1560.186990
Эпоха 600, MSE: 1707.266640
Эпоха 700, MSE: 1879.127582


KeyboardInterrupt: 

In [4]:
import math
import csv
import numpy as np

class NeuralNetwork:
    def __init__(self, activationFunc, deactivationFunc, hiddenWeights: list, outputWeights: list, requestedEra: int):
        self.requestedEra = requestedEra
        self.hiddenWeights = hiddenWeights
        self.outputWeights = outputWeights
        self.activationFunc = activationFunc
        self.deactivationFunc = deactivationFunc

        self.MseNumerator = 0.0
        self.CurrentMSE = 0.0

        self.E = 0.001     # Скорость обучения
        self.a = 0.1       # Момент

        self.dataset = []
        self.normalized_dataset = []
        self.answersList = []
        
        # Инициализация дельт
        self.hiddenWeightDeltas = [[0.0, 0.0, 0.0] for _ in range(3)]
        self.outputWeightDeltas = [0.0, 0.0, 0.0]
        
        # Для логарифмической нормализации
        self.input_mins = None
        self.input_maxs = None
        self.output_min = None
        self.output_max = None
        self.input_means = None
        self.input_stds = None

    def readFromCsv(self, datasetPath: str):
        with open(datasetPath, 'r') as csvDataset:
            csvDatasetReader = csv.reader(csvDataset, delimiter='\t')
            next(csvDatasetReader)  # Пропускаем заголовок
            for row in csvDatasetReader:
                self.dataset.append([float(row[0]), float(row[1]), float(row[2]), float(row[3])])
        
        # Логарифмическая нормализация данных
        self._log_normalizeDataset()

    def _log_normalizeDataset(self):
        """Логарифмическая нормализация для улучшения экстраполяции"""
        if not self.dataset:
            return
            
        # Отделяем входы и выходы
        inputs = np.array([row[:3] for row in self.dataset])
        outputs = np.array([row[3] for row in self.dataset])
        
        # Логарифмическое преобразование входов (добавляем 1 чтобы избежать log(0))
        log_inputs = np.log1p(inputs)  # log(1 + x)
        
        # Нормализуем логарифмированные входы в [-1, 1]
        self.input_means = np.mean(log_inputs, axis=0)
        self.input_stds = np.std(log_inputs, axis=0)
        
        # Для выходов используем обычную нормализацию в [-1, 1]
        self.output_min = np.min(outputs)
        self.output_max = np.max(outputs)
        
        # Применяем нормализацию
        self.normalized_dataset = []
        for i, row in enumerate(self.dataset):
            # Логарифмическая нормализация входов
            normalized_inputs = [
                (log_inputs[i][j] - self.input_means[j]) / self.input_stds[j]
                for j in range(3)
            ]
            
            # Обычная нормализация выходов в [-1, 1]
            if self.output_max != self.output_min:
                normalized_output = 2 * (row[3] - self.output_min) / (self.output_max - self.output_min) - 1
            else:
                normalized_output = 0
            
            self.normalized_dataset.append(normalized_inputs + [normalized_output])
        
        print(f"Логарифмическая нормализация применена")
        print(f"Входы: log(1+x) -> стандартизация")
        print(f"Выходы: {self.output_min:.2f} - {self.output_max:.2f} -> -1 - 1")

    def normalize_input(self, input_values):
        """Нормализация новых входных значений для предсказания"""
        log_inputs = np.log1p(input_values)
        normalized = [
            (log_inputs[i] - self.input_means[i]) / self.input_stds[i]
            for i in range(3)
        ]
        return normalized

    def denormalize_output(self, normalized_output):
        """Денормализация выхода"""
        return self.output_min + (normalized_output + 1) * (self.output_max - self.output_min) / 2

    def train(self):
        era_mse = []
        best_mse = float('inf')
        patience = 50
        patience_counter = 0
        
        for era in range(self.requestedEra):
            self.MseNumerator = 0.0
            self.answersList = []
            
            # Перемешиваем данные каждую эпоху
            np.random.shuffle(self.normalized_dataset)
            
            for fullTrainSet in self.normalized_dataset:
                ideal = fullTrainSet[-1]
                trainSet = fullTrainSet[0:-1]

                # Прямое распространение
                hiddenNeuronInputsValues = self._calculateHiddenInputsValues(trainSet)
                hiddenNeuronOutputsValues = self._calculateHiddenOutputsValues(hiddenNeuronInputsValues)
                output = self._calculateOutput(hiddenNeuronOutputsValues)

                # Денормализованный вывод для MSE
                denormalized_output = self.denormalize_output(output)
                denormalized_ideal = self.denormalize_output(ideal)
                
                self.answersList.append(denormalized_output)
                self.MseNumerator += ((denormalized_ideal - denormalized_output) ** 2)

                # Обратное распространение
                output_error = ideal - output
                dO = output_error * self.deactivationFunc(output)

                # Вычисление dH
                dH = []
                for i in range(len(hiddenNeuronOutputsValues)):
                    dH_i = self.deactivationFunc(hiddenNeuronOutputsValues[i]) * dO * self.outputWeights[i]
                    dH.append(dH_i)

                # Обновление весов
                self.outputWeightDeltas = self._calculateOutputWeightDeltas(dO, hiddenNeuronOutputsValues)
                self.hiddenWeightDeltas = self._calculateHiddenWeightDeltas(trainSet, dH)

                self.hiddenWeights = self._calculateNewHiddenWeigths(self.hiddenWeights, self.hiddenWeightDeltas)
                self.outputWeights = self._calculateNewOutputWeigths(self.outputWeights, self.outputWeightDeltas)

            # Вычисление MSE для эпохи
            epoch_mse = self.MseNumerator / len(self.dataset)
            era_mse.append(epoch_mse)
            
            # Early stopping
            if epoch_mse < best_mse:
                best_mse = epoch_mse
                patience_counter = 0
                best_weights = (self.hiddenWeights.copy(), self.outputWeights.copy())
            else:
                patience_counter += 1
            
            if era % 100 == 0:
                print(f"Эпоха {era}, MSE: {epoch_mse:.6f}, LR: {self.E:.6f}")
                
            # Уменьшаем скорость обучения если нет улучшений
            if patience_counter > 20:
                self.E *= 0.95
                patience_counter = 0
                
            if patience_counter >= patience:
                print(f"Ранняя остановка на эпохе {era}, лучший MSE: {best_mse:.6f}")
                self.hiddenWeights, self.outputWeights = best_weights
                break
        
        return era_mse

    def predict(self, a, b, c):
        """Предсказание для новых данных"""
        # Нормализуем вход
        normalized_input = self.normalize_input([a, b, c])
        
        # Прямое распространение
        hidden_inputs = self._calculateHiddenInputsValues(normalized_input)
        hidden_outputs = self._calculateHiddenOutputsValues(hidden_inputs)
        output = self._calculateOutput(hidden_outputs)
        
        # Денормализуем выход
        return self.denormalize_output(output)

    def _calculateHiddenInputsValues(self, trainSet: list) -> list:
        result = []
        for weights in self.hiddenWeights:
            halfResult = 0.0
            for i in range(len(weights)):
                halfResult += trainSet[i] * weights[i]
            result.append(halfResult)
        return result

    def _calculateHiddenOutputsValues(self, inputs: list) -> list:
        result = []
        for inputValue in inputs:
            result.append(self.activationFunc(inputValue))
        return result
    
    def _calculateOutput(self, hiddenOutputsValues: list) -> float:
        result = 0.0
        for i in range(len(hiddenOutputsValues)):
            result += hiddenOutputsValues[i] * self.outputWeights[i]
        return result

    def _calculateOutputWeightDeltas(self, dO: float, hiddenNeuronOutputsValues: list) -> list:
        outputDeltas = []
        for i in range(len(hiddenNeuronOutputsValues)):
            outputGradient = dO * hiddenNeuronOutputsValues[i]
            previousOutputDelta = self.outputWeightDeltas[i]
            delta = self.E * outputGradient + self.a * previousOutputDelta
            outputDeltas.append(delta)
        return outputDeltas

    def _calculateHiddenWeightDeltas(self, trainSet: list, dH: list) -> list:
        hiddenDeltas = []
        for i in range(len(dH)):
            deltasOfLine = []
            for j in range(len(trainSet)):
                hiddenGradient = trainSet[j] * dH[i]
                previousHiddenDelta = self.hiddenWeightDeltas[i][j]
                delta = self.E * hiddenGradient + self.a * previousHiddenDelta
                deltasOfLine.append(delta)
            hiddenDeltas.append(deltasOfLine)
        return hiddenDeltas

    def _calculateNewHiddenWeigths(self, currentWeights: list, deltas: list):
        newWeights = []
        for i in range(len(currentWeights)):
            lineOfNewWeights = []
            for j in range(len(currentWeights[i])):
                lineOfNewWeights.append(currentWeights[i][j] + deltas[i][j])
            newWeights.append(lineOfNewWeights)
        return newWeights

    def _calculateNewOutputWeigths(self, currentWeights: list, deltas: list):
        newWeights = []
        for i in range(len(currentWeights)):
            newWeights.append(currentWeights[i] + deltas[i])
        return newWeights

# Активационные функции
def f_tang(x):
    return math.tanh(x)

def f_anti_tang(x):
    return 1 - x**2

# Ваши веса
hiddenWeights = [[0.1, 0.9, 0.31], [0.4, 0.7, 0.11], [0.3, 0.2, 0.27]]
outputWeights = [0.47, 0.51, 0.67]

print("Используем ваши веса:")
print("Hidden:", hiddenWeights)
print("Output:", outputWeights)

# Создаем и обучаем сеть
myAI = NeuralNetwork(f_tang, f_anti_tang, hiddenWeights, outputWeights, 5000)
myAI.readFromCsv("discriminant_data.csv")
mse_history = myAI.train()

# Тестируем на новых данных
print("\nТестирование экстраполяции:")
test_cases = [
    (1, 2, 1),    # D = 0
    (1, 5, 1),    # D = 21
    (2, 4, 1),    # D = 8
    (0.5, 3, 2),  # D = 5
    (4, 4, 4),    # D = -48
]

for a, b, c in test_cases:
    predicted = myAI.predict(a, b, c)
    actual = b**2 - 4*a*c
    error = abs(predicted - actual)
    print(f"a={a}, b={b}, c={c} -> Предсказано: {predicted:.3f}, Реальное: {actual:.3f}, Ошибка: {error:.3f}")

Используем ваши веса:
Hidden: [[0.1, 0.9, 0.31], [0.4, 0.7, 0.11], [0.3, 0.2, 0.27]]
Output: [0.47, 0.51, 0.67]
Логарифмическая нормализация применена
Входы: log(1+x) -> стандартизация
Выходы: -91.83 - 23.24 -> -1 - 1
Эпоха 0, MSE: nan, LR: 0.001000


<ipython-input-4-8f17b7209df8>:126: RuntimeWarning: overflow encountered in double_scalars
  self.MseNumerator += ((denormalized_ideal - denormalized_output) ** 2)
<ipython-input-4-8f17b7209df8>:247: RuntimeWarning: overflow encountered in double_scalars
  return 1 - x**2
<ipython-input-4-8f17b7209df8>:135: RuntimeWarning: invalid value encountered in double_scalars
  dH_i = self.deactivationFunc(hiddenNeuronOutputsValues[i]) * dO * self.outputWeights[i]


Эпоха 100, MSE: nan, LR: 0.000815
Эпоха 200, MSE: nan, LR: 0.000630
Эпоха 300, MSE: nan, LR: 0.000488
Эпоха 400, MSE: nan, LR: 0.000377
Эпоха 500, MSE: nan, LR: 0.000307
Эпоха 600, MSE: nan, LR: 0.000238
Эпоха 700, MSE: nan, LR: 0.000184
Эпоха 800, MSE: nan, LR: 0.000142
Эпоха 900, MSE: nan, LR: 0.000116
Эпоха 1000, MSE: nan, LR: 0.000090
Эпоха 1100, MSE: nan, LR: 0.000069
Эпоха 1200, MSE: nan, LR: 0.000054
Эпоха 1300, MSE: nan, LR: 0.000044
Эпоха 1400, MSE: nan, LR: 0.000034
Эпоха 1500, MSE: nan, LR: 0.000026
Эпоха 1600, MSE: nan, LR: 0.000020
Эпоха 1700, MSE: nan, LR: 0.000017
Эпоха 1800, MSE: nan, LR: 0.000013
Эпоха 1900, MSE: nan, LR: 0.000010
Эпоха 2000, MSE: nan, LR: 0.000008
Эпоха 2100, MSE: nan, LR: 0.000006
Эпоха 2200, MSE: nan, LR: 0.000005
Эпоха 2300, MSE: nan, LR: 0.000004
Эпоха 2400, MSE: nan, LR: 0.000003
Эпоха 2500, MSE: nan, LR: 0.000002
Эпоха 2600, MSE: nan, LR: 0.000002
Эпоха 2700, MSE: nan, LR: 0.000001
Эпоха 2800, MSE: nan, LR: 0.000001
Эпоха 2900, MSE: nan, LR: 0.0